# bce-log-loss-real-fake — ex2: discriminator loss via BCE-with-logits (numerically stable form)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bce-log-loss-real-fake`. Running the final beacon cell reports progress against the `GAN: BCE log loss real/fake` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: BCE log loss real/fake` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bce-log-loss-real-fake`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bce-log-loss-real-fake"
DD_SUBTOPIC = "GAN: BCE log loss real/fake"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BCE-with-logits — numerically stable D loss — deepening

Ex1 used `F.binary_cross_entropy(probs, targets)` — the version that expects post-sigmoid probabilities. In production GAN code you should feed LOGITS to `F.binary_cross_entropy_with_logits` instead — it fuses `sigmoid + bce` internally with the log-sum-exp trick:

```python
loss_real = F.binary_cross_entropy_with_logits(D_logits_real, t.ones_like(D_logits_real))
loss_fake = F.binary_cross_entropy_with_logits(D_logits_fake, t.zeros_like(D_logits_fake))
loss_D = loss_real + loss_fake
```

**Why this matters.** When D becomes very confident on a batch (logits at ±20), the sigmoid output is `1 - 1e-9` or `1e-9`, and `log(1 - p)` or `log(p)` plus float32 rounding crash to `inf` (or `nan`) in the bare `binary_cross_entropy` path. The fused form sidesteps that — see `softplus(x) = log(1 + exp(x))` for the exact algebra.

**Numerical equivalence.** For non-extreme logits (in `[-10, +10]`-ish), the logits form and the probs form agree to single-precision.

### Exercise 2 — discriminator loss via BCE-with-logits (numerically stable form)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `F.binary_cross_entropy_with_logits` with ones/zeros targets to compute the discriminator loss directly from logits, and verify numerical equivalence to the probs form at moderate logits plus stability advantage at extreme logits.
> Keywords: bce-with-logits, numerical-stability, gan, logits-form
> ```

**KCs targeted:** `bce-log-loss-real-fake`, `bce-with-logits-fused`

Implement `ex2_discriminator_loss_logits(d_logits_real, d_logits_fake)`. The numerically-stable form of the discriminator loss:

1. `d_logits_real` are D's RAW LOGIT outputs on real images (shape `(B,)`, ANY real-valued float — pre-sigmoid).
2. `d_logits_fake` are D's raw logits on fakes (same shape, same values range).
3. Build target tensors:
   - `real_t = t.ones_like(d_logits_real)`
   - `fake_t = t.zeros_like(d_logits_fake)`
4. Compute `loss_real = F.binary_cross_entropy_with_logits(d_logits_real, real_t)`.
5. Compute `loss_fake = F.binary_cross_entropy_with_logits(d_logits_fake, fake_t)`.
6. Return `loss_real + loss_fake` — a scalar tensor.

**Do NOT call sigmoid then `F.binary_cross_entropy`.** The whole point is to keep the operation in logit-space so the fused kernel uses `softplus` instead of `log(sigmoid(x))`. The test stresses this with logits at ±50 where the unfused form is `inf`/`nan`.

Returns a scalar tensor.

In [ ]:
def ex2_discriminator_loss_logits(d_logits_real: Tensor, d_logits_fake: Tensor) -> Tensor:
    import torch.nn.functional as F
    loss_real = F.binary_cross_entropy_with_logits(d_logits_real, t.ones_like(d_logits_real))
    loss_fake = F.binary_cross_entropy_with_logits(d_logits_fake, t.zeros_like(d_logits_fake))
    return loss_real + loss_fake


<details><summary>Solution</summary>

```python
def ex2_discriminator_loss_logits(d_logits_real: Tensor, d_logits_fake: Tensor) -> Tensor:
    import torch.nn.functional as F
    loss_real = F.binary_cross_entropy_with_logits(d_logits_real, t.ones_like(d_logits_real))
    loss_fake = F.binary_cross_entropy_with_logits(d_logits_fake, t.zeros_like(d_logits_fake))
    return loss_real + loss_fake
```

**Why the fused form is stable.** `bce_with_logits(x, 1) = -log(sigmoid(x)) = softplus(-x)`, and `softplus` is implemented as `log1p(exp(-|x|)) + max(x, 0)` — the absolute-value trick guarantees `exp` always sees a non-positive argument. So `softplus(50) ≈ 50` (computed exactly) while `-log(sigmoid(50)) = -log(1 - 1e-22)` underflows to `inf` in float32.

**When to use which.** Always prefer `binary_cross_entropy_with_logits` in production. The plain `binary_cross_entropy` is occasionally useful for plotting losses against PREDICTED probabilities (interpretability), but never inside the training loop.

**Gradient is also stable.** `d/dx bce_with_logits(x, target) = sigmoid(x) - target`, computed without ever materializing `1 - sigmoid(x)` near 1.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()